In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Misurare i qubit

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Per ottenere informazioni sullo stato di un qubit, puoi _misurarlo_ su un [bit classico](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit). In Qiskit, le misurazioni vengono eseguite nella base computazionale, ovvero la base di Pauli-$Z$ a singolo qubit. Di conseguenza, una misurazione restituisce 0 o 1, in base alla sovrapposizione con gli autostati di Pauli-$Z$ $|0\rangle$ e $|1\rangle$:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## Misurazioni a metà circuito
Le misurazioni a metà circuito sono una componente fondamentale dei circuiti dinamici. Prima di `qiskit-ibm-runtime` v0.43.0, `measure` era l'unica istruzione di misurazione in Qiskit. Le misurazioni a metà circuito, tuttavia, hanno requisiti di calibrazione diversi rispetto alle misurazioni _terminali_ (misurazioni che avvengono alla fine di un circuito). Ad esempio, bisogna considerare la durata dell'istruzione quando si calibra una misurazione a metà circuito, poiché istruzioni più lunghe generano circuiti più rumorosi. Non è necessario considerare la durata delle istruzioni per le misurazioni terminali, poiché non ci sono istruzioni successive.

In `qiskit-ibm-runtime` v0.43.0 è stata introdotta l'istruzione `MidCircuitMeasure`. Come suggerisce il nome, si tratta di una nuova istruzione di misurazione ottimizzata per le misurazioni a metà circuito su QPU IBM&reg;.

> **Note:** L'istruzione `MidCircuitMeasure` corrisponde all'istruzione `measure_2` riportata nelle `supported_instructions` del backend. Tuttavia, `measure_2` non è supportata su tutti i backend. Usa `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` per trovare i backend che la supportano. In futuro potrebbero essere aggiunte nuove misurazioni, ma non è garantito.

## Applicare una misurazione a un circuito
Esistono diversi modi per applicare misurazioni a un circuito:

### Metodo `QuantumCircuit.measure`
Usa il metodo [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure) per misurare un [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class).

Esempi:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### Classe `Measure`
La classe Qiskit [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) misura i qubit specificati.

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### Metodo `QuantumCircuit.measure_all`
Per misurare tutti i qubit nei corrispondenti bit classici, usa il metodo [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all). Per impostazione predefinita, questo metodo aggiunge nuovi bit classici in un `ClassicalRegister` per memorizzare le misurazioni.